## QMEGS - Quantum Multiple Eigenvalue Gaussian filtered Search

This notebook demonstrates the QMEGS algorithm for estimating multiple eigenvalues
of a Hamiltonian using Hadamard tests and Gaussian filtering.

**Reference:** [arXiv:2402.01013](https://arxiv.org/abs/2402.01013)

In [ ]:
import numpy as np
import qarpx as qx
from qarp.operators import FullyCommuting, QubitOperator
from qarp.operators.functions import eigenspectrum

from qarp.blocks import TrotterBlock, SimpleBlock, SynthesizedTimeEvolutionBlock
from qarp.algorithms import QMEGS, get_overlaps

### 1. Define the Hamiltonian

We use a simple 2-qubit Hamiltonian with known eigenvalues for demonstration.

In [ ]:
# Define a 2-qubit Hamiltonian
hamiltonian = (
    QubitOperator("Z0 X1") + QubitOperator("Y0 Y1") + QubitOperator("X0 X1")
) * 0.25

n_qubits = 2

# Print the exact eigenspectrum for reference
spectrum = eigenspectrum(hamiltonian)
print("Exact eigenvalues:", spectrum)

### 2. Prepare the Unitary and Trial State

QMEGS requires:
- A **unitary block** implementing time evolution under H (via Trotterization)
- A **trial state** with significant overlap with the target eigenstates

In [ ]:
# Build the Trotter unitary for time evolution exp(-iHt)
# unitary_block = TrotterBlock(
#     n_qubits=n_qubits,
#     operator=hamiltonian,
#     steps=10,
#     order=4,
#     grouping=FullyCommuting()
# )

# Alternatively, use a synthesized time evolution block to get a more compact circuit and to remove trotterization errors
# Note, do not build the block yet, pass it directly to QMEGS
unitary_block = SynthesizedTimeEvolutionBlock(
    n_qubits=n_qubits,
    operator=hamiltonian,
    )

# Create a simple trial state: |0⟩ ⊗ |+⟩
trial_state = SimpleBlock(n_qubits).h(1)
trial_state.build().plot()

# Check overlaps with eigenstates to verify trial state quality
overlaps, eigenvalues = get_overlaps(trial_state, hamiltonian, return_eigenvalues=True)
print("Overlaps with eigenstates:", overlaps)
print("Eigenvalues:", eigenvalues)

### 3. Configure and Run QMEGS

QMEGS requires specifying which eigenvalues to target via `target_indices` (indices of dominant eigenvalues).
The algorithm requires `pmin > ptail` where:
- `pmin` = minimum overlap among target eigenstates
- `ptail` = sum of overlaps with non-target eigenstates

In [ ]:
# Select the two eigenstates with highest overlap
sorted_indices = np.argsort(overlaps)[::-1]
target_indices = sorted_indices[:2].tolist()

# Verify the overlap condition is satisfied
pmin = min(overlaps[i] for i in target_indices)
ptail = sum(overlaps[i] for i in range(len(overlaps)) if i not in target_indices)
print(f"Target indices: {target_indices}")
print(f"Target eigenvalues: {eigenvalues[target_indices]}")
print(f"pmin: {pmin:.4f}, ptail: {ptail:.4f}, pmin > ptail: {pmin > ptail}")

In [ ]:
# Create and run the QMEGS algorithm
qmegs = QMEGS(
    unitary=unitary_block,          # Block implementing time evolution
    state=trial_state,              # Trial state having overlap with target eigenstates
    n_shots=None,                   # Use statevector simulation
    target_indices=target_indices,  # Target eigenstate indices
    sigma=1.0,                      # Truncation level
    eta=0.01,                       # Precision parameter
    T=100,                          # Time window
    mode_dataset='sampling',        # 'analytical' or 'sampling'
    overlaps='classical',           # O(4^n) diagonalisation + exact trial statevector: a validation step
    mode_time='rvs',                # 'rvs' or 'rejection_sampling' (rejection_sampling allows to specify the filtering function)
    verbose=True,
).build()

### Hardware-shaped form: supply the overlaps

`overlaps='classical'` diagonalises the Hamiltonian to obtain `p_min` / `p_tail` — the a-priori inputs Theorem 1 of the paper assumes known — and is refused on a noisy or routed engine. When those two numbers are known (or bounded) independently, pass them directly: no spectrum is computed, and the sampling path runs on any engine. Only `len(target_indices)` is used then (how many eigenvalues to extract).


In [ ]:
import qarpx as qx
from qarp.algorithms import HadamardTest
from qarp.devices import NoiseModel
from qarp.engines import QarpEngine

noisy_engine = QarpEngine(
    n_qubits=trial_state.n_qubits + 1,  # +1: HadamardTest's compiled circuit needs an ancilla
    noise_model=NoiseModel.depolarizing(1e-3, [qx.GateType.CX]),
    seed=0,
)
qmegs_hw = QMEGS(
    unitary=unitary_block,
    state=trial_state,
    n_shots=2000,
    target_indices=target_indices,
    T=100,
    mode_dataset='sampling',
    overlaps=(pmin, ptail),         # from the cell above; no diagonalisation inside QMEGS
    primitive=HadamardTest(),
    engine=noisy_engine,
).build()
print('Found eigenphases (noisy engine):', qmegs_hw.run())


In [ ]:
# Run the algorithm and get estimated eigenvalues
result = qmegs.run()

print("Estimated eigenvalues:", sorted(result))
print("Exact eigenvalues:    ", sorted(eigenvalues[target_indices]))

### 4. Visualize the Density Function

QMEGS uses a filtered density function to identify eigenvalue peaks.

In [ ]:
import matplotlib.pyplot as plt

# Generate samples and compute the filtered density function
dataset = qmegs._generate_data_analytical()
# dataset = qmegs._generate_data_sampling()

J = int(np.floor(2 * np.pi * qmegs.T / qmegs.q))
theta_js = np.array([-np.pi + j * qmegs.q / qmegs.T for j in range(J + 1)])
G_js = qmegs.filtered_density_function(dataset, theta_js, qmegs.n_samples)

# Plot the density function with true eigenvalue markers
plt.figure(figsize=(10, 4))
plt.plot(theta_js, G_js, label='Filtered density G(θ)')
for i, eig in enumerate(eigenvalues[target_indices]):
    plt.axvline(x=eig, color='r', linestyle='--', alpha=0.7,
                label=f'True λ_{i}={eig:.4f}')
plt.xlabel('θ')
plt.ylabel('G(θ)')
plt.title('QMEGS Filtered Density Function')
plt.legend()
plt.tight_layout()
plt.show()

### Compare the measurement results with the analytical evolution

In [ ]:
import matplotlib.pyplot as plt

# Generate test times
t_range = (-80, 80)

n_points_measurement = 50
test_times_measurement = np.linspace(t_range[0], t_range[1], n_points_measurement)

n_points_analytical = 400
test_times_analytical = np.linspace(t_range[0], t_range[1], n_points_analytical)

# Analytical expectation: <ψ|U(t)|ψ> = Σ_k p_k * e^(-i*λ_k*t)
# QMEGS always uses exp(-iHt) convention (after reversal if needed)
analytical = qmegs.p.dot(np.exp(-1j * np.outer(qmegs.eigenvalues, test_times_analytical)))

# Circuit measurements using the primitive
circuit_vals = []
for t in test_times_measurement:
    try:
        z = qmegs._perform_measurement(t, verbose=True)
        circuit_vals.append(z)
    except Exception as e:
        print(f"Measurement failed at t={t}: {e}")
        circuit_vals.append(np.nan)

# Plot comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.plot(test_times_analytical, np.real(analytical), "b-", label="Analytical Re")
ax1.scatter(test_times_measurement, np.real(circuit_vals), c="r", label="Circuit Re", alpha=0.7)
ax1.set_xlabel("Time")
ax1.set_ylabel("Re(<ψ|U(t)|ψ>)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(test_times_analytical, np.imag(analytical), "b-", label="Analytical Im")
ax2.scatter(test_times_measurement, np.imag(circuit_vals), c="r", label="Circuit Im", alpha=0.7)
ax2.set_xlabel("Time")
ax2.set_ylabel("Im(<ψ|U(t)|ψ>)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()